In [0]:
dbutils.library.restartPython()

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from typing import Any

spark: Any
dbutils: Any

# CS4603 PA4 — Document Analyst

Development & testing notebook. Section headers match the tasks in `README.md`.
Fill in each cell, run everything top-to-bottom, and **keep all outputs visible** before submitting.
Record explanations and analysis answers in `STUDENT_ANALYSIS.md`.


## Part 0 — Setup & Corpus Ingestion
Env config + ingest `data/annual_report.pdf` into Databricks Vector Search (Task 0.3).


In [0]:
%pip install -e .

In [0]:
# TODO(0.1): load config / verify env vars
import os
import sys
import importlib.util
from pathlib import Path

# Prevent bytecode caching which fails on workspace paths
sys.dont_write_bytecode = True

# Load config module manually (workspace import quirk)
REPO_ROOT = Path("/Workspace/Users/27100380@lums.edu.pk/27100380_cs4603-pa4").resolve()
config_file = REPO_ROOT / "config.py"
spec = importlib.util.spec_from_file_location("config", config_file)
config = importlib.util.module_from_spec(spec)
sys.modules['config'] = config
spec.loader.exec_module(config)

from config import get_settings

# Task 0.1: configure and verify the non-secret Databricks resource names.
# Change these values only if you used different names during setup.

os.environ["SOURCE_TABLE"] = "cs4603.default.s27100380_analyst_chunks"
os.environ["VECTOR_SEARCH_ENDPOINT"] = "s27100380-vs-endpoint"
os.environ["VECTOR_SEARCH_INDEX"] = "cs4603.default.s27100380_analyst_index"
os.environ["EMBEDDINGS_ENDPOINT"] = "databricks-gte-large-en"

VOLUME_PATH = "/Volumes/cs4603/default/pa4/annual_report.pdf"

# Load rag package manually
rag_init = REPO_ROOT / "rag" / "__init__.py"
spec = importlib.util.spec_from_file_location("rag", rag_init)
rag = importlib.util.module_from_spec(spec)
rag.__path__ = [str(REPO_ROOT / "rag")]
sys.modules['rag'] = rag
spec.loader.exec_module(rag)

from rag.ingest import build_chunks_table

build_chunks_table(
    spark,
    volume_path="/Volumes/cs4603/default/pa4/annual_report.pdf",
    chunks_table="cs4603.default.s27100380_analyst_chunks",
)

for name in (
    "SOURCE_TABLE",
    "VECTOR_SEARCH_ENDPOINT",
    "VECTOR_SEARCH_INDEX",
    "EMBEDDINGS_ENDPOINT",
):
    assert os.environ.get(name), f"Missing {name}"
    print(f"{name}={os.environ[name]}")

display(dbutils.fs.ls("/Volumes/cs4603/default/pa4/"))


In [0]:
# TODO(0.3): ingest corpus -> Delta table -> Vector Search index; wait until READY
# from rag.ingest import ingest
# ingest(spark, volume_path='/Volumes/main/default/pa4/annual_report.pdf')
# Task 0.3: parse, chunk, create/sync the index, wait for READY, and test it.
from rag.ingest import ingest

index = ingest(spark, volume_path=VOLUME_PATH)

display(
    spark.sql(
        f"""
        SELECT chunk_id, source, page, chunk_to_retrieve, chunk_to_embed
        FROM {os.environ['SOURCE_TABLE']}
        ORDER BY page
        """
    )
)

index.describe()



In [0]:
from rag.ingest import verify_index
print(index.describe())
verify_index(
    index,
    query="What was the net income in 2023?",
)

In [0]:
display(
    spark.sql("""
        SELECT chunk_id, source, page, chunk_to_retrieve
        FROM cs4603.default.s27100380_analyst_chunks
        ORDER BY page
    """)
)

## Part 1 — Build the Document Analyst graph
Nodes: planner (1.2), supervisor (1.3), RAG agent (1.4), MCP tools (1.5), synthesizer (1.6), full graph (1.7).


In [0]:
import os

os.environ["DATABRICKS_HOST"] = "https://dbc-005bf88c-4348.cloud.databricks.com"
os.environ["DATABRICKS_MODEL"] = "databricks-meta-llama-3-3-70b-instruct"

os.environ["EMBEDDINGS_ENDPOINT"] = "databricks-gte-large-en"
os.environ["VECTOR_SEARCH_ENDPOINT"] = "s27100380-vs-endpoint"
os.environ["VECTOR_SEARCH_INDEX"] = "cs4603.default.s27100380_analyst_index"

In [0]:
import sys
import importlib

sys.modules.pop("agent.graph", None)
sys.modules.pop("rag.store", None)
importlib.invalidate_caches()

import rag.store
import agent.graph

In [0]:
import sys
import importlib.util
from pathlib import Path

# Load agent package manually (workspace import quirk)
REPO_ROOT = Path("/Workspace/Users/27100380@lums.edu.pk/27100380_cs4603-pa4").resolve()
agent_init = REPO_ROOT / "agent" / "__init__.py"
spec = importlib.util.spec_from_file_location("agent", agent_init)
agent = importlib.util.module_from_spec(spec)
agent.__path__ = [str(REPO_ROOT / "agent")]
sys.modules['agent'] = agent
spec.loader.exec_module(agent)

from agent.graph import build_graph

graph = build_graph()
print("Graph compiled successfully.")

In [0]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [0]:
import sys
import importlib

for module in [
    "agent.graph",
    "agent.planner",
    "agent.supervisor",
    "agent.rag_agent",
    "agent.synthesizer",
]:
    sys.modules.pop(module, None)

importlib.invalidate_caches()

from agent.graph import build_graph

graph = build_graph()

In [0]:
import sys
import importlib

sys.modules.pop("agent.graph", None)
importlib.invalidate_caches()

from agent.graph import build_graph

graph = build_graph()

### Test the graph


In [0]:
retrieval_result = graph.invoke({
    "messages": [{
        "role": "user",
        "content": "What was the net income in 2023?"
    }]
})

print("Plan:", retrieval_result["plan"])
print("Step results:", retrieval_result["step_results"])
print("Final answer:")
print(retrieval_result["messages"][-1].content)

In [0]:
# Computation-only query
computation_result = graph.invoke({
    "messages": [{
        "role": "user",
        "content": "What is 15% of 2.4 billion?"
    }]
})

print("Plan:", computation_result["plan"])
print("Step results:", computation_result["step_results"])
print("Final answer:")
print(computation_result["messages"][-1].content)


In [0]:
combined_input = {
    "messages": [{
        "role": "user",
        "content": (
            "What was the revenue in 2023, "
            "and what would a 10% increase look like?"
        )
    }]
}

combined_result = graph.invoke(combined_input)

print("Plan:", combined_result["plan"])
print("Step results:", combined_result["step_results"])
print("Final answer:")
print(combined_result["messages"][-1].content)

In [0]:
for event in graph.stream(combined_input, stream_mode="updates"):
    for node_name, update in event.items():
        print(f"\n{'=' * 15} {node_name} {'=' * 15}")

        if "plan" in update:
            print("Plan:", update["plan"])

        if "next_agent" in update:
            print("Route:", update["next_agent"])

        if "step_results" in update:
            print("Step results:", update["step_results"])

        if "current_step_index" in update:
            print("Next step index:", update["current_step_index"])

        if "final_answer" in update:
            print("Final answer:", update["final_answer"])

### Required — offline smoke test
Runs the graph with a mocked LLM (no Databricks calls). Same test Bonus A automates.


In [0]:
import sys
import importlib

sys.modules.pop("agent.graph", None)
importlib.invalidate_caches()

In [0]:
%sh PYTHONPYCACHEPREFIX=/tmp/pa4_pycache python -m pytest tests/test_smoke.py -q -o cache_dir=/tmp/pa4_pytest_cache

## Part 2 — Deployment
Package as an MLflow models-from-code model, register in Unity Catalog, create the serving endpoint (Tasks 2.1–2.4).
Reference: `databricks_deployment_v1/deployment.ipynb`.


In [0]:
# TODO(2.1): sanity-check the model definition imports cleanly
!python -c "import deployment.agent_model"



In [0]:
import os

os.environ["UC_CATALOG"] = "cs4603"
os.environ["UC_SCHEMA"] = "default"
os.environ["UC_MODEL_NAME"] = "s27100380_document_analyst"

In [0]:
import sys
import importlib

sys.modules.pop("deployment.deploy", None)
importlib.invalidate_caches()

from deployment.deploy import log_and_register

uc_model_name, model_version = log_and_register()
print(uc_model_name, model_version)


In [0]:
from getpass import getpass
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceAlreadyExists

w = WorkspaceClient()
scope = "cs4603-deploy"

try:
    w.secrets.create_scope(scope=scope)
except ResourceAlreadyExists:
    pass

w.secrets.put_secret(
    scope=scope,
    key="DATABRICKS_HOST",
    string_value="https://dbc-005bf88c-4348.cloud.databricks.com",
)

w.secrets.put_secret(
    scope=scope,
    key="DATABRICKS_TOKEN",
    string_value=getpass("Enter your all-apis Databricks token: "),
)

w.secrets.put_secret(
    scope=scope,
    key="DATABRICKS_MODEL",
    string_value="databricks-meta-llama-3-3-70b-instruct",
)

print([secret.key for secret in w.secrets.list_secrets(scope)])

In [0]:

import os

os.environ["VECTOR_SEARCH_ENDPOINT"] = "s27100380-vs-endpoint"
os.environ["VECTOR_SEARCH_INDEX"] = "cs4603.default.s27100380_analyst_index"
os.environ["EMBEDDINGS_ENDPOINT"] = "databricks-gte-large-en"
os.environ["SERVING_ENDPOINT_NAME"] = "s27100380-document-analyst"
os.environ["SECRET_SCOPE"] = "cs4603-deploy"


import importlib
import deployment.deploy

importlib.reload(deployment.deploy)


os.environ["UC_CATALOG"] = "cs4603"
os.environ["UC_SCHEMA"] = "default"
os.environ["UC_MODEL_NAME"] = "s27100380_document_analyst"
uc_name, new_version = deployment.deploy.log_and_register()
print(uc_name, new_version)


In [0]:
import importlib
import deployment.deploy

importlib.reload(deployment.deploy)

uc_name, new_version = deployment.deploy.log_and_register()
print(uc_name, new_version)

In [0]:
deployment.deploy.create_or_update_endpoint(
    uc_name,
    new_version,
)

### Test the deployed endpoint (Task 2.4)


In [0]:
import json
import subprocess
import time

import requests

DATABRICKS_HOST = dbutils.secrets.get(
    "cs4603-deploy",
    "DATABRICKS_HOST",
).rstrip("/")

DATABRICKS_TOKEN = dbutils.secrets.get(
    "cs4603-deploy",
    "DATABRICKS_TOKEN",
)

ENDPOINT_NAME = "s27100380-document-analyst"

url = (
    f"{DATABRICKS_HOST}/serving-endpoints/"
    f"{ENDPOINT_NAME}/invocations"
)

print("Endpoint:", ENDPOINT_NAME)
print("Invocation URL:", url)

In [0]:
curl_payload = json.dumps({
    "messages": [
        {
            "role": "user",
            "content": "What was the net income in 2023?",
        }
    ]
})

curl_start = time.perf_counter()

curl_result = subprocess.run(
    [
        "curl",
        "--silent",
        "--show-error",
        "--fail-with-body",
        "--max-time", "600",
        "--request", "POST",
        url,
        "--header", f"Authorization: Bearer {DATABRICKS_TOKEN}",
        "--header", "Content-Type: application/json",
        "--data", curl_payload,
    ],
    capture_output=True,
    text=True,
)

curl_latency = time.perf_counter() - curl_start

print("RAW CURL RESPONSE:")
print(curl_result.stdout)

if curl_result.returncode != 0:
    print("CURL ERROR:")
    print(curl_result.stderr)
    raise RuntimeError("curl request failed")

print(f"\nCurl request latency: {curl_latency:.2f} seconds")

In [0]:
r = requests.post(
    url,
    headers={
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json",
    },
    json={
        "messages": [
            {
                "role": "user",
                "content": "What was the net income in 2023?",
            }
        ]
    },
    timeout=600,
)

print("HTTP status:", r.status_code)
r.raise_for_status()

raw_response = r.json()

print("\nRAW RESPONSE:")
print(json.dumps(raw_response, indent=2))

parsed_answer = raw_response[0]["messages"][-1]["content"]

print("\nPARSED ANSWER:")
print(parsed_answer)

In [0]:
def call_deployed_endpoint(query):
    start = time.perf_counter()

    response = requests.post(
        url,
        headers={
            "Authorization": f"Bearer {DATABRICKS_TOKEN}",
            "Content-Type": "application/json",
        },
        json={
            "messages": [
                {
                    "role": "user",
                    "content": query,
                }
            ]
        },
        timeout=600,
    )

    latency = time.perf_counter() - start
    response.raise_for_status()

    raw = response.json()
    answer = raw[0]["messages"][-1]["content"]

    return {
        "query": query,
        "raw": raw,
        "answer": answer,
        "latency": latency,
    }

In [0]:
test_queries = [
    "What was the net income in 2023?",
    "What is 15% of 2.4 billion?",
    (
        "What was the revenue in 2023, "
        "and what would a 10% increase look like?"
    ),
]

deployed_results = []

for query in test_queries:
    result = call_deployed_endpoint(query)
    deployed_results.append(result)

    print("QUERY:")
    print(result["query"])

    print("\nDEPLOYED ANSWER:")
    print(result["answer"])

    print(f"\nLATENCY: {result['latency']:.2f} seconds")
    print("=" * 80)

In [0]:
comparison_results = []

for deployed in deployed_results:
    query = deployed["query"]

    local_state = graph.invoke({
        "messages": [
            {
                "role": "user",
                "content": query,
            }
        ]
    })

    local_last_message = local_state["messages"][-1]

    if hasattr(local_last_message, "content"):
        local_answer = local_last_message.content
    else:
        local_answer = local_last_message["content"]

    deployed_answer = deployed["answer"]

    comparison_results.append({
        "query": query,
        "local_answer": local_answer,
        "deployed_answer": deployed_answer,
        "identical": (
            local_answer.strip() == deployed_answer.strip()
        ),
    })

for result in comparison_results:
    print("QUERY:")
    print(result["query"])

    print("\nLOCAL ANSWER:")
    print(result["local_answer"])

    print("\nDEPLOYED ANSWER:")
    print(result["deployed_answer"])

    print("\nEXACTLY IDENTICAL:", result["identical"])
    print("=" * 80)

Comparison explanation:
The `identical` field checks strict string equality, so even a small difference
in wording, punctuation, formatting, or citation placement produces `False`.

The local and deployed responses may not be exactly identical because LLM output
is not guaranteed to be the exact same even when the temperature
is set to zero. The deployed request is executed in a separate serving
container, and minor differences in retrieval ordering
or generated phrasing affects the final text.

The comparison is via semantic consistency. For the
retrieval-only query, both responses should report the same document fact and
citation. For the computation-only query, both should produce the same numeric
result. For the combined query, both should use the same retrieved revenue value
and calculate the same 10% increase. so differing wording is acceptable
as long as the facts, calculations, and citations are the same

In [0]:
latency_query = "What was the net income in 2023?"

cold_result = call_deployed_endpoint(latency_query)
warm_result = call_deployed_endpoint(latency_query)

print(f"Cold/first request: {cold_result['latency']:.2f} seconds")
print(f"Warm request:       {warm_result['latency']:.2f} seconds")
print(
    "Latency difference: "
    f"{cold_result['latency'] - warm_result['latency']:.2f} seconds"
)

The cold request was slower because Databricks had to load the model. The warm request reused the active model
container. If the endpoint had not actually scaled to zero before the first
request, the result represents first-request versus warm latency rather than cold start

## Part 3 — Client SDK demo
Instantiate `DocumentAnalystClient`, health-check, ask, stream, and show timeout/retry handling (Task 3.2).


In [0]:
import sys
import importlib

# Clear cached module to force fresh import with updated code
if 'client' in sys.modules:
    del sys.modules['client']
if 'client.sdk' in sys.modules:
    del sys.modules['client.sdk']
importlib.invalidate_caches()

from client.sdk import (
    AnalystClientError,
    DocumentAnalystClient,
)

DATABRICKS_HOST = dbutils.secrets.get(
    "cs4603-deploy",
    "DATABRICKS_HOST",
)

DATABRICKS_TOKEN = dbutils.secrets.get(
    "cs4603-deploy",
    "DATABRICKS_TOKEN",
)

client = DocumentAnalystClient(
    endpoint_name="s27100380-document-analyst",
    host=DATABRICKS_HOST,
    token=DATABRICKS_TOKEN,
    timeout=120.0,
    max_retries=3,
    
)

healthy = client.health_check()

print("Endpoint healthy:", healthy)
assert healthy is True

answer = client.ask("What was the net income in 2023?")

print("\nASK RESULT:")
print(answer)

In [0]:
print("STREAMING RESULT:\n")

for chunk in client.ask_streaming(
    "Summarize the FY2023 revenue."
):
    print(chunk, end="", flush=True)

print()

In [0]:
from unittest.mock import patch

import requests

timeout_client = DocumentAnalystClient(
    endpoint_name="s27100380-document-analyst",
    host=DATABRICKS_HOST,
    token=DATABRICKS_TOKEN,
    timeout=0.001,
    max_retries=0,
)

with patch.object(
    timeout_client._session,
    "request",
    side_effect=requests.Timeout("Simulated timeout"),
):
    try:
        timeout_client.ask("What was the revenue in 2023?")
    except TimeoutError as exc:
        print("TIMEOUT HANDLED CORRECTLY:")
        print(exc)

In [0]:
import json
from unittest.mock import call, patch

import requests


def simulated_response(status_code, body, request_id):
    response = requests.Response()
    response.status_code = status_code
    response._content = json.dumps(body).encode("utf-8")
    response.headers["Content-Type"] = "application/json"
    response.headers["x-databricks-request-id"] = request_id
    return response


retry_client = DocumentAnalystClient(
    endpoint_name="s27100380-document-analyst",
    host=DATABRICKS_HOST,
    token=DATABRICKS_TOKEN,
    timeout=120.0,
    max_retries=2,
)

unavailable_responses = [
    simulated_response(
        503,
        {"message": "Endpoint temporarily unavailable"},
        "retry-test-1",
    ),
    simulated_response(
        503,
        {"message": "Endpoint temporarily unavailable"},
        "retry-test-2",
    ),
    simulated_response(
        503,
        {"message": "Endpoint still unavailable"},
        "retry-test-3",
    ),
]

with (
    patch.object(
        retry_client._session,
        "request",
        side_effect=unavailable_responses,
    ) as request_mock,
    patch("client.sdk.time.sleep") as sleep_mock,
):
    try:
        retry_client.ask("What was the revenue in 2023?")
    except AnalystClientError as exc:
        print("RETRY FAILURE HANDLED CORRECTLY")
        print("Status code:", exc.status_code)
        print("Message:", str(exc))
        print("Request ID:", exc.request_id)

    print("\nTotal HTTP attempts:", request_mock.call_count)
    print("Backoff calls:", sleep_mock.call_args_list)

    assert request_mock.call_count == 3
    assert sleep_mock.call_args_list == [call(1.0), call(2.0)]